# EDA & feature engineering — Seguro de Vida Individual

Partimos de los datos limpios en `data/processed/clean/` — resultado de la extracción
de fuentes CNSF 2021–2024 y la promoción de encabezados reales documentada en
`data_cleaning.ipynb`.

Este notebook cubre tres etapas:

1. **Inspección** — tipos, rangos, nulos y distribuciones por tabla.
2. **Join inferido** — identificar la clave compuesta que permite cruzar `emision`,
   `siniestros` y `comisiones` sin foreign key explícito.
3. **Feature engineering** — construir métricas por segmento (siniestralidad,
   frecuencia, severidad, prima por asegurado) como inputs válidos para modelos de
   pricing.

Ver `docs/ANTECEDENTES.md` para contexto completo del proyecto.

---
## 0. Carga de datos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN = Path("../data/processed/clean")

emision_f    = pd.read_csv(CLEAN / "emision_clean.csv",    dtype=str)
siniestros_f = pd.read_csv(CLEAN / "siniestros_clean.csv", dtype=str)
comisiones_f = pd.read_csv(CLEAN / "comisiones_clean.csv", dtype=str)

frames = {"emision": emision_f, "siniestros": siniestros_f, "comisiones": comisiones_f}

for name, df in frames.items():
    print(f"✓ {name:<12} {df.shape}")

✓ emision      (3508592, 12)
✓ siniestros   (297504, 13)
✓ comisiones   (827906, 16)


---
## 1. Inspección — esquema de columnas y tipos raw

Todas las columnas llegaron como `str`. Aquí vemos qué existe en cada tabla
antes de cualquier casteo.

In [2]:
for name, df in frames.items():
    print(f"\n{'='*60}")
    print(f"  {name.upper()}  — {df.shape[0]:,} filas x {df.shape[1]} columnas")
    print(f"{'='*60}")
    print(df.dtypes.to_string())


  EMISION  — 3,508,592 filas x 12 columnas
2021                      str
EDAD                      str
COBERTURA                 str
PLAN DE LA POLIZA         str
MODALIDAD DE LA POLIZA    str
MONEDA                    str
ENTIDAD                   str
SEXO                      str
FORMA DE VENTA            str
NUMERO DE ASEGURADOS      str
PRIMA EMITIDA             str
SUMA ASEGURADA            str

  SINIESTROS  — 297,504 filas x 13 columnas
2021                      str
EDAD                      str
COBERTURA                 str
ENTIDAD                   str
CAUSA DEL SINIESTRO       str
PLAN DE LA POLIZA         str
MODALIDAD DE LA POLIZA    str
SEXO                      str
NUMERO DE SINIESTROS      str
MONTO RECLAMADO           str
VENCIMIENTOS              str
MONTO PAGADO              str
MONTO DE REASEGURO        str

  COMISIONES  — 827,906 filas x 16 columnas
2021                       str
EDAD                       str
PLAN DE LA POLIZA          str
MODALIDAD DE LA POLIZA 

---
## 2. Casteo de tipos

Separamos las columnas en dos grupos por tabla:
- **Categóricas / clave**: `ANIO`, `ENTIDAD`, `SEXO`, `EDAD`, `COBERTURA`, `CLAVE_*`, etc.
- **Numéricas**: primas, sumas aseguradas, siniestros, comisiones, número de asegurados.

Usamos `errors='coerce'` para convertir cadenas inválidas en `NaN` y detectarlas.

In [ ]:
# Reconoziced true headres
for name, df in frames.items():
    print(f"\n[{name}]")
    print(list(df.columns))

In [5]:
# ── 1. Renombrar y limpiar nombres de columna ───────────────────────────────
def normalize_columns(df):
    """Strip espacios, reemplaza espacios internos por _, uppercase."""
    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
        .str.replace(' ', '_', regex=False)
    )
    # La primera columna es ANIO (quedó como '2021' por el header del Excel)
    first_col = df.columns[0]
    if first_col != 'ANIO':
        df = df.rename(columns={first_col: 'ANIO'})
    return df

for name in frames:
    frames[name] = normalize_columns(frames[name])

emision_f    = frames["emision"]
siniestros_f = frames["siniestros"]
comisiones_f = frames["comisiones"]

# Verificar
for name, df in frames.items():
    print(f"\n[{name}]")
    print(list(df.columns))


[emision]
['ANIO', 'EDAD', 'COBERTURA', 'PLAN_DE_LA_POLIZA', 'MODALIDAD_DE_LA_POLIZA', 'MONEDA', 'ENTIDAD', 'SEXO', 'FORMA_DE_VENTA', 'NUMERO_DE_ASEGURADOS', 'PRIMA_EMITIDA', 'SUMA_ASEGURADA']

[siniestros]
['ANIO', 'EDAD', 'COBERTURA', 'ENTIDAD', 'CAUSA_DEL_SINIESTRO', 'PLAN_DE_LA_POLIZA', 'MODALIDAD_DE_LA_POLIZA', 'SEXO', 'NUMERO_DE_SINIESTROS', 'MONTO_RECLAMADO', 'VENCIMIENTOS', 'MONTO_PAGADO', 'MONTO_DE_REASEGURO']

[comisiones]
['ANIO', 'EDAD', 'PLAN_DE_LA_POLIZA', 'MODALIDAD_DE_LA_POLIZA', 'MONEDA', 'ENTIDAD', 'SEXO', 'FORMA_DE_VENTA', 'TIPO_DIVIDENDO', 'NUMERO_DE_ASEGURADOS', 'PRIMA_CEDIDA', 'COMISIONES_DIRECTAS', 'FONDO_DE_INVERSIÓN', 'FONDO_DE_ADMINISTRACION', 'MONTO_DE_DIVIDENDOS', 'MONTO_DE_RESCATE']


In [6]:
# ── Columnas numéricas conocidas por tabla ──────────────────────────────────
NUM_COLS = {
    "emision": [
        "PRIMA_EMITIDA",
        "SUMA_ASEGURADA",
        "NUMERO_DE_ASEGURADOS",
    ],
    "siniestros": [
        "NUMERO_DE_SINIESTROS",
        "MONTO_RECLAMADO",
        "MONTO_PAGADO",
        "VENCIMIENTOS",
        "MONTO_DE_REASEGURO",
    ],
    "comisiones": [
        "NUMERO_DE_ASEGURADOS",
        "PRIMA_CEDIDA",
        "COMISIONES_DIRECTAS",
        "FONDO_DE_INVERSIÓN",
        "FONDO_DE_ADMINISTRACION",
        "MONTO_DE_DIVIDENDOS",
        "MONTO_DE_RESCATE",
    ],
}

def cast_numerics(df, num_cols):
    """Castea a float las columnas numéricas que existan en df."""
    existing = [c for c in num_cols if c in df.columns]
    missing  = [c for c in num_cols if c not in df.columns]
    if missing:
        print(f"  ⚠️  Columnas no encontradas: {missing}")
    for c in existing:
        df[c] = pd.to_numeric(df[c].str.strip().str.replace(',', '', regex=False),
                              errors='coerce')
    return df

for name, df in frames.items():
    print(f"\n[{name}]")
    frames[name] = cast_numerics(df, NUM_COLS.get(name, []))

# Reasignar variables individuales
emision_f    = frames["emision"]
siniestros_f = frames["siniestros"]
comisiones_f = frames["comisiones"]

print("\n✓ Casteo completado")


[emision]

[siniestros]

[comisiones]

✓ Casteo completado


In [8]:
# ── Persistir datasets normalizados ─────────────────────────────────────────
CLEAN_V2 = Path("../data/processed/clean_v2")
CLEAN_V2.mkdir(parents=True, exist_ok=True)

for name, df in frames.items():
    path = CLEAN_V2 / f"{name}_v2.csv"
    df.to_csv(path, index=False)
    print(f"✓ {name:<12} → {path}  {df.shape}")

✓ emision      → ../data/processed/clean_v2/emision_v2.csv  (3508592, 12)
✓ siniestros   → ../data/processed/clean_v2/siniestros_v2.csv  (297504, 13)
✓ comisiones   → ../data/processed/clean_v2/comisiones_v2.csv  (827906, 16)


---
## ⚡ Quick start — sesiones posteriores
Cargar desde `clean_v2/` si el pipeline de normalización ya fue ejecutado.
Saltar las secciones 0–2 si se parte desde aquí.

In [ ]:
import pandas as pd
from pathlib import Path

CLEAN_V2 = Path("../data/processed/clean_v2")

emision_f    = pd.read_csv(CLEAN_V2 / "emision_v2.csv")
siniestros_f = pd.read_csv(CLEAN_V2 / "siniestros_v2.csv")
comisiones_f = pd.read_csv(CLEAN_V2 / "comisiones_v2.csv")

frames = {"emision": emision_f, "siniestros": siniestros_f, "comisiones": comisiones_f}

for name, df in frames.items():
    print(f"✓ {name:<12} {df.shape}")a

---
## 3. Estadísticas descriptivas — columnas numéricas

Para cada tabla: `count`, `mean`, `std`, `min`, `p25`, `p50`, `p75`, `p95`, `max`.

El percentil 95 es clave para detectar colas largas típicas en seguros.

In [7]:
PERCENTILES = [0.25, 0.50, 0.75, 0.95]

for name, df in frames.items():
    num_df = df.select_dtypes(include='number')
    if num_df.empty:
        print(f"\n[{name}] — sin columnas numéricas casteadas")
        continue
    desc = num_df.describe(percentiles=PERCENTILES).T
    desc.columns = [c.upper() for c in desc.columns]   # estandarizar nombres
    print(f"\n{'='*70}")
    print(f"  {name.upper()} — estadísticas numéricas")
    print(f"{'='*70}")
    print(desc.to_string(float_format=lambda x: f"{x:,.2f}"))


  EMISION — estadísticas numéricas
                            COUNT          MEAN            STD            MIN       25%        50%          75%           95%               MAX
NUMERO_DE_ASEGURADOS 3,508,589.00         60.00         518.18           1.00      2.00       5.00        23.00        257.00        130,071.00
PRIMA_EMITIDA        3,508,589.00    248,091.69   3,198,567.96 -14,894,612.05    448.32   4,394.01    37,568.77    635,092.05  2,448,270,922.42
SUMA_ASEGURADA       3,508,589.00 19,452,081.65 102,688,613.09           0.00 60,008.43 995,821.00 6,469,399.00 81,257,198.08 66,326,216,079.25

  SINIESTROS — estadísticas numéricas
                          COUNT       MEAN          STD             MIN    25%       50%        75%          95%            MAX
NUMERO_DE_SINIESTROS 297,501.00       2.40        10.64            1.00   1.00      1.00       2.00         7.00       3,304.00
MONTO_RECLAMADO      297,501.00 249,207.14 1,922,788.72  -61,776,356.79   0.01 34,000.00 200,

---
## 4. Diagnóstico de nulos

Reportamos el porcentaje de nulos por columna en cada tabla. Un nulo en una
columna numérica post-casteo puede indicar:
- valor original vacío (`''`), o
- cadena no parseable (e.g. `'N/D'`, `'-'`, valores con formato incorrecto).

Ambos casos requieren decisión antes del join o feature engineering.

In [ ]:
def null_report(df, name):
    total = len(df)
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / total * 100).round(2)
    report = pd.DataFrame({
        "nulos": null_counts,
        "pct_%": null_pct,
        "dtype": df.dtypes,
    })
    report = report[report["nulos"] > 0].sort_values("pct_%", ascending=False)
    print(f"\n{'='*55}")
    print(f"  {name.upper()} — columnas con nulos ({total:,} filas total)")
    print(f"{'='*55}")
    if report.empty:
        print("  ✅ Sin nulos detectados")
    else:
        print(report.to_string())

for name, df in frames.items():
    null_report(df, name)

---
## 5. Cardinalidad y valores únicos — columnas categóricas

Para las columnas tipo `object` (posibles claves de segmentación), reportamos:
- cardinalidad (# valores únicos),
- los top-10 valores por frecuencia.

Esto permite identificar qué columnas son candidatas a clave compuesta de join.

In [ ]:
def categorical_report(df, name, top_n=10):
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    print(f"\n{'='*60}")
    print(f"  {name.upper()} — columnas categóricas ({len(cat_cols)} cols)")
    print(f"{'='*60}")
    for col in cat_cols:
        vc = df[col].value_counts(dropna=False)
        card = df[col].nunique(dropna=False)
        null_n = df[col].isnull().sum()
        print(f"\n  ▸ {col}  (cardinalidad={card:,}, nulos={null_n:,})")
        print(vc.head(top_n).to_string())

for name, df in frames.items():
    categorical_report(df, name)

---
## 6. Cobertura temporal — distribución por ANIO

Verificamos que los cuatro años (2021–2024) estén representados de forma
homogénea en las tres tablas. Asimetrías pueden indicar datos faltantes
o diferencias en el reporte por año.

In [ ]:
anio_col = "ANIO"   # ajustar si el nombre difiere

print(f"{'TABLA':<15} {'ANIO':<8} {'FILAS':>10} {'PCT':>8}")
print("-" * 45)

for name, df in frames.items():
    if anio_col not in df.columns:
        print(f"  ⚠️  {name}: columna '{anio_col}' no encontrada")
        continue
    vc = df[anio_col].value_counts().sort_index()
    total = len(df)
    for anio, cnt in vc.items():
        pct = cnt / total * 100
        print(f"{name:<15} {str(anio):<8} {cnt:>10,} {pct:>7.1f}%")
    print("-" * 45)

---
## 7. Columnas compartidas — candidatas a clave compuesta

Identificamos las columnas presentes en las tres tablas simultáneamente.
Son las candidatas naturales para el join inferido del siguiente bloque.

In [ ]:
col_sets = {name: set(df.columns) for name, df in frames.items()}

shared_all   = col_sets["emision"] & col_sets["siniestros"] & col_sets["comisiones"]
shared_em_si = col_sets["emision"] & col_sets["siniestros"]
shared_em_co = col_sets["emision"] & col_sets["comisiones"]
shared_si_co = col_sets["siniestros"] & col_sets["comisiones"]

print("Columnas en las TRES tablas:")
print(sorted(shared_all))

print("\nColumnas sólo en emision ∩ siniestros (no en comisiones):")
print(sorted(shared_em_si - col_sets["comisiones"]))

print("\nColumnas sólo en emision ∩ comisiones (no en siniestros):")
print(sorted(shared_em_co - col_sets["siniestros"]))

print("\nColumnas exclusivas por tabla:")
for name, cs in col_sets.items():
    excl = cs - (shared_em_si | shared_em_co | shared_si_co)
    print(f"  {name}: {sorted(excl)}")

---
## 8. Resumen ejecutivo — criterio de homogeneidad

Con base en los resultados anteriores evaluamos si las tres tablas corresponden
a la **misma cartera** y si la granularidad permite construir features válidos.

Criterios de homogeneidad:

| Criterio | Indicador |
|---|---|
| Cobertura temporal | Mismos años en las tres tablas |
| Entidades reportantes | Mismo conjunto de `ENTIDAD` |
| Segmentación demográfica | `SEXO`, `EDAD`, `COBERTURA` consistentes |
| Escala numérica | Rangos razonables para seguros de vida individual MXN |
| Nulos | Ausencia o patrón sistemático explicable |


In [ ]:
# ── Comparación de entidades entre tablas ───────────────────────────────────
entidad_col = "ENTIDAD"   # ajustar si el nombre difiere

entidades = {}
for name, df in frames.items():
    if entidad_col in df.columns:
        entidades[name] = set(df[entidad_col].dropna().unique())

if len(entidades) == 3:
    names = list(entidades.keys())
    common = entidades[names[0]] & entidades[names[1]] & entidades[names[2]]
    print(f"Entidades comunes a las 3 tablas: {len(common)}")
    for name, ents in entidades.items():
        solo = ents - common
        print(f"  {name}: {len(ents)} entidades totales, {len(solo)} exclusivas → {sorted(solo) if solo else '—'}")
else:
    print("⚠️  Columna ENTIDAD no encontrada en alguna tabla — revisar nombre")

# ── Resumen de cardinalidad de claves demográficas ──────────────────────────
print()
demo_cols = ["SEXO", "EDAD", "COBERTURA"]
print(f"{'COLUMNA':<15}", end="")
for name in frames:
    print(f"  {name:>12}", end="")
print()
print("-" * (15 + 15 * len(frames)))

for col in demo_cols:
    print(f"{col:<15}", end="")
    for name, df in frames.items():
        if col in df.columns:
            card = df[col].nunique()
            print(f"  {card:>12,}", end="")
        else:
            print(f"  {'—':>12}", end="")
    print()